# Agent Smith · Kaggle GPU run
Attach a private Kaggle Dataset containing the **Agent_Smith_Kaggle.zip** release. Enable Internet and a GPU accelerator. Run the cells in order.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

WORK = Path("/kaggle/working")
PROJECT = WORK / "Agent_Smith_Kaggle"
matches = sorted(Path("/kaggle/input").rglob("Agent_Smith_Kaggle.zip"))
if len(matches) != 1:
    raise RuntimeError(f"Attach exactly one dataset containing Agent_Smith_Kaggle.zip. Found {matches}")
with ZipFile(matches[0]) as archive:
    if any(not (WORK / name).resolve().is_relative_to(WORK) for name in archive.namelist()):
        raise ValueError("Invalid path in release ZIP")
    archive.extractall(WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT / "requirements.txt")], check=True)

SEEDS = [17, 42]
EPOCHS = 3
BATCH = 16
RUN = WORK / "agent_smith_run"

env = {**os.environ, "PYTHONHASHSEED": "0", "TOKENIZERS_PARALLELISM": "false"}
def run(script, *args):
    subprocess.run([sys.executable, "-u", script, *map(str, args)], cwd=PROJECT, env=env, check=True)

flags = ["--out", RUN, "--epochs", EPOCHS, "--batch", BATCH, "--seed", *SEEDS]


### 1. Check CUDA and the three-class checkpoint

In [ ]:
run("smoke.py")

### 2. Train and freeze

In [ ]:
run("kaggle_run.py", "train", *flags)

### 3. Evaluate on validation and held-out test

In [ ]:
run("kaggle_run.py", "evaluate", *flags)

### 4. Export results and print the held-out summary

In [ ]:
import json
import tempfile

for seed in SEEDS:
    metrics = json.loads((RUN / f"seed_{seed}" / "test_metrics.json").read_text())
    for variant, row in metrics["variants"].items():
        print(seed, variant, "accuracy", round(row["full_accuracy"]["rate"], 4),
              "unsafe allows", row["unsafe_allows"]["n"], "/", row["unsafe_allows"]["d"])
if (RUN / "live_metrics.json").exists():
    live = json.loads((RUN / "live_metrics.json").read_text())
    for condition, row in live["conditions"].items():
        print(condition, "correct", row["correct"]["n"], "/", row["correct"]["d"])

with tempfile.TemporaryDirectory(dir=WORK) as temp:
    shutil.copytree(RUN, Path(temp) / RUN.name, ignore=shutil.ignore_patterns("input"))
    bundle = shutil.make_archive(str(WORK / "Agent_Smith_Kaggle_Results"), "zip", temp)
print(bundle)
